# Algorytmika i matematyka uczenia maszynowego 
## Laboratorium 11

### Zadanie 1

Zaimplementuj systemu rekomendacji filmów w oparciu o indeks Jaccarda. System ma za zadanie zwrócić listę filmów sugerowany dla podanego użytownika.

Dane zostały pobrane z serwisu Kaggle z https://www.kaggle.com/datasets/gargmanas/movierecommenderdataset

Zbiór zawiera dwa pliki:
- `movies.csv` lista filmów wraz z ich identyfikatorami
- `ratings.csv` lista ocen filmów przez użytkowników

**Zadanie:**
* Wczytaj oba pliki.
* Zamień wszystkie oceny użytkownika na wartość 1 (zastosuj próg okreśjący czy film się podobał czy nie np. 3).
* Stwórz macierz ocen użytkowników w której wierszach będą użytkownicy, a w kolumnach filmy. Wartość w macierzy jest flagą mówiącą czy użytkownikowi film się podobał czy nie. 
* Wypełnij brakujące wartości zerami.
* Utwórz macierz podobieństwa Jaccarda pomiędzy użytkownikami (każdy z każdym).
    - Możesz wykorzystać funkcję [jaccard](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.jaccard.html) z biblioteki scipy.
* Zaimplementuj funkcję która dla podanego użytkownika zwróci listę sugerowanych filmów.
    - Funkcja powinna zwrócić listę filmów które nie były ocenione przez użytkownika, a które są rekomendowane dla niego.



In [3]:
import pandas as pd
from scipy.spatial.distance import pdist, squareform, jaccard

movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

ratings['rating'] = (ratings['rating'] > 2.5).astype('int')

user_movie_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating', fill_value=0)

jaccard_distances = pdist(user_movie_matrix.values, metric='jaccard')
jaccard_similarity_matrix = 1 - squareform(jaccard_distances)

def get_recommendations(user_id, top_n=5):
    user_index = user_movie_matrix.index.get_loc(user_id)
    user_similarity_scores = jaccard_similarity_matrix[user_index]
    
    similar_users_indices = user_similarity_scores.argsort()[::-1][1:]
    similar_users_scores = user_similarity_scores[similar_users_indices]
    
    similar_users_ratings = user_movie_matrix.iloc[similar_users_indices]
    
    weighted_ratings = similar_users_ratings.T.dot(similar_users_scores)
    already_rated = user_movie_matrix.loc[user_id]
    weighted_ratings = weighted_ratings[already_rated == 0]
    weighted_ratings /= similar_users_scores.sum()
    
    recommendations = weighted_ratings[weighted_ratings > 0].sort_values(ascending=False).head(top_n)
    
    recommended_movies = movies[movies['movieId'].isin(recommendations.index)].copy()
    recommended_movies['predicted_rating'] = recommendations.values
    
    return recommended_movies[['movieId', 'title', 'predicted_rating']]

for user_id in [1, 2]:
    print(f"Recommendations for User {user_id}:")
    recommendations = get_recommendations(user_id)
    print(recommendations)
    print("\n")


Recommendations for User 1:
      movieId                                      title  predicted_rating
31         32  Twelve Monkeys (a.k.a. 12 Monkeys) (1995)          0.609685
277       318           Shawshank Redemption, The (1994)          0.524415
507       589          Terminator 2: Judgment Day (1991)          0.434107
659       858                      Godfather, The (1972)          0.416967
2078     2762                    Sixth Sense, The (1999)          0.412464


Recommendations for User 2:
      movieId                             title  predicted_rating
257       296               Pulp Fiction (1994)          0.596796
314       356               Forrest Gump (1994)          0.596599
510       593  Silence of the Lambs, The (1991)          0.571359
1939     2571                Matrix, The (1999)          0.509655
2226     2959                 Fight Club (1999)          0.485196






### Zadanie 2


Algorytm MinHash na przykładzie wykrywania plagiatów

Wykonaj kolejno następujące kroki:

1. Pobierz 8 akapitów tekstu (nie za krótkich), każdy o różnej tematyce (mogą być np. z różnych haseł Wikipedii), trzymaj się jednego języka (np. PL lub ENG). Wklej je do jednego pliku tekstowego, z linią wolną jako separatorem.

2. Skopiuj wybrane 2-3 akapity i ręcznie nieco zmodyfikuj.

> Przykład (z hasła https://pl.wikipedia.org/wiki/Fryderyk_Chopin):

```Jest uważany za jednego z najwybitniejszych kompozytorów romantycznych, a także za jednego z najważniejszych polskich kompozytorów w historii. Był jednym z najsłynniejszych pianistów swoich czasów, często nazywany poetą fortepianu. Elementem charakterystycznym dla utworów Chopina jest pogłębiona ekspresja oraz czerpanie z wzorców stylistycznych polskiej muzyki ludowej.```

↓↓↓ Zmieniono na ↓↓↓

```Jest uważany za jednego z najwybitniejszych kompozytorów romantycznych, a także za jednego z najważniejszych kompozytorów polskich w historii.  Był jednym  z najsłynniejszych pianistów swoich czasów, często nazywany poetą fortepianu! Elementem charakterystycznym dla utworów Fryderyka Chopina jest pogłębiona ekspresja oraz czerpanie z wzorców polskiej muzyki ludowej.```

Otrzymasz zatem w pliku tekstowym 10 lub 11 akapitów tekstu (kolejność dowolna, te „splagiatowane” nie muszą być na końcu).

3. Z poziomu skryptu: wczytaj wszystkie akapity z pliku. Zbuduj 100 "losowych" funkcji haszujących.

> Sugestia: funkcją "bazową" jest po prostu `hash(...)`. Zakładamy 64-bitową wersję Pythona 3.x, wtedy `hash(...)` jest 64-bitowy.

Na liście seeds umieszczamy 100 losowych liczb 64-bitowych. Aby obliczyć $i$-ty hash dla ciągu 
należy wykonać `hash(s) ^ seeds[i]` (użycie operatora XOR).

4. Przyjmij niewielką wartość $Q$ (np. 15) i dla każdego akapitu
    - oznacz jego długość przez $n$,
    - dla każdej ze 100 funkcji haszujących policz hasza w przesuwnym oknie tekstu o długości $Q$ znaków (czyli łącznie mamy $n - Q + 1$ wartości hasza); zapamiętaj MINIMUM z tych $n - Q + 1$
 wartości.
Na wyjściu mamy zatem (dla 11 akapitów) 11 * 100 wartości haszy.

5. Rozważ pary akapitów "każdy z każdym". Jeśli dla danej pary co najmniej (np.) 30 haszy jest wspólnych, to uważamy akapity za podobne (być może plagiat) i wyświetlamy na ekranie.

6. Wyświetl czas obliczeń (powinien wynosić mniej niż 0.5s).

7. Poeksperymentuj z liczbą użytych funkcji haszujących, wartością, stopniem modyfikacji oryginalnych akapitów tekstu, progiem detekcji akapitów podobnych.

In [9]:
import hashlib
import random
import time

Q = 15            
NUM_HASHES = 100      
PLAGIAT_THRESHOLD = 30

with open("akapit.txt", "r", encoding="utf-8") as f:
    paragraphs = [p.strip() for p in f.read().split("\n\n") if p.strip()]

def stable_hash(s, seed):
    h = int(hashlib.sha1(s.encode()).hexdigest(), 16)
    return h ^ seed

random.seed(123)
seeds = [random.getrandbits(64) for _ in range(NUM_HASHES)]

def minhash_signature(text):
    n = len(text)
    sig = []
    for seed in seeds:
        min_h = float('inf')
        for i in range(n - Q + 1):
            window = text[i:i + Q]
            h = stable_hash(window, seed)
            if h < min_h:
                min_h = h
        sig.append(min_h)
    return sig

start = time.time()
signatures = [minhash_signature(p) for p in paragraphs]

print("Podejrzane pary akapitów (potencjalny plagiat):\n")
for i in range(len(signatures)):
    for j in range(i + 1, len(signatures)):
        wspolne = sum(1 for a, b in zip(signatures[i], signatures[j]) if a == b)
        if wspolne >= PLAGIAT_THRESHOLD:
            print(f"Akapit {i+1} i {j+1} → wspólnych hashy: {wspolne}")

print(f"\nCzas wykonania: {time.time() - start:.3f}s")


Podejrzane pary akapitów (potencjalny plagiat):

Akapit 3 i 11 → wspólnych hashy: 100

Czas wykonania: 0.305s
